# Workshop AI Summit - IA Multimodal con Snowflake Cortex

**¡Bienvenidos!** En 20 minutos vamos a descubrir juntos cómo Snowflake procesa **imágenes, documentos y audio** con IA nativa, y cómo es posible crear **agentes conversacionales realmente potentes** usando solo lenguaje natural con **Cortex Code**.

## Qué vamos a hacer (paso a paso)
1. Vamos a analizar imágenes con `AI_COMPLETE` multimodal
2. Después, vamos a pedirle a Snowflake que extraiga datos de contratos en DOCX
3. Vamos a transcribir llamadas de servicio al cliente y mediremos cómo se sintió el cliente
4. Y cerramos creando, **desde cero y en español**, un agente conversacional con Cortex Code

> **Antes de empezar:** verifiquemos arriba a la derecha del notebook que tengamos seleccionado el warehouse `HOL_WH` y el schema `HOL_AI_SUMMIT.PUBLIC`. Si no, lo cambiamos. ¿Listos? ¡Vamos!

## Ejercicio 1: Imágenes con IA Multimodal

¿Sabían que Snowflake puede *ver* imágenes y entenderlas? Vamos a probarlo. Le vamos a dar a `AI_COMPLETE` una foto de un siniestro vehicular y le pediremos que actúe como **perito de seguros**. Después, le pasaremos una imagen de una cédula y le pediremos que extraiga los datos como si fuera un proceso de KYC.

Spoiler: lo va a hacer increíblemente bien. 🎯

In [ ]:
-- Que imágenes tenemos en el stage?
SELECT RELATIVE_PATH, SIZE FROM DIRECTORY(@IMAGENES) ORDER BY RELATIVE_PATH;

In [ ]:
-- Analizar imagen del siniestro vehicular con IA multimodal
SELECT AI_COMPLETE(
  'claude-4-sonnet',
  'Eres un perito de seguros. Describe el daño del vehiculo en esta imagen, indica severidad (leve/moderado/grave) y estima un rango de costo de reparacion en USD. Responde en español y en formato JSON con campos: descripcion, severidad, costo_estimado.',
  TO_FILE('@IMAGENES', 'choque.png')
) AS analisis_siniestro;

In [ ]:
-- Extraer datos de identificacion de una cédula (KYC)
SELECT AI_COMPLETE(
  'claude-4-sonnet',
  'Extrae los datos de esta cédula en formato JSON: numero_documento, nombres, apellidos, fecha_nacimiento, lugar_expedicion. Si no puedes leer un campo, devuelve null.',
  TO_FILE('@IMAGENES', 'cédula.jpg')
) AS datos_cédula;

## Ejercicio 2: Documentos con AI_PARSE_DOCUMENT y AI_EXTRACT

Aquí viene algo poderoso. Snowflake parsea **DOCX, PDF, PPTX y más formatos** sin que tengamos que pre-procesar nada. Vamos a ver cómo en una sola consulta convertimos contratos legales en datos estructurados que cualquier sistema puede consumir.

Imaginen cuántas horas se ahorrarían los equipos legales y de operaciones con esto.

In [ ]:
-- Ver los contratos ya parseados (creados por setup.sql)
SELECT file_name, LEFT(content, 300) AS preview FROM DOCS_PARSED;

In [ ]:
-- Extraer campos clave de los contratos. Fíjate en la columna numero_poliza_seguro:
-- ese ID conecta el contrato con la tabla POLIZAS del Workshop.
SELECT
  file_name,
  AI_EXTRACT(
    content,
    ['arrendador', 'arrendatario', 'valor_canon_mensual', 'plazo_meses', 'direccion_inmueble', 'fecha_inicio', 'numero_poliza_seguro', 'aseguradora']
  ) AS campos_extraidos
FROM DOCS_PARSED;

In [ ]:
-- Generar un resumen ejecutivo de cada contrato
SELECT
  file_name,
  AI_SUMMARIZE_AGG(content) AS resumen
FROM DOCS_PARSED
GROUP BY file_name;

## Ejercicio 3: Audio con AI_TRANSCRIBE, AI_SENTIMENT y AI_COMPLETE

Este es uno de mis favoritos. Vamos a tomar dos llamadas reales de servicio al cliente, las vamos a transcribir, vamos a medir el sentimiento del cliente y, lo más importante, le vamos a pedir a Snowflake que **le dé recomendaciones al asesor** para mejorar su próxima interacción.

**Cada función con su propósito:**
- `AI_SENTIMENT` → mide el sentimiento (general y por aspectos como producto, precio, asesor)
- `AI_CLASSIFY` → identifica el motivo de la llamada
- `AI_COMPLETE` → genera coaching personalizado para el asesor

¿Se imaginan tener esto corriendo sobre 10.000 llamadas todas las noches? Eso es lo que va a poder hacer nuestra organización.

In [ ]:
-- Ver transcripciones (generadas por setup.sql con AI_TRANSCRIBE)
SELECT file_name, LEFT(transcripcion, 250) AS preview
FROM TRANSCRIPCIONES;

In [ ]:
-- Análisis de sentimiento con AI_SENTIMENT (categorías y polaridad)
SELECT
  file_name,
  AI_SENTIMENT(transcripcion):categories[0]:sentiment::VARCHAR AS sentimiento_general,
  AI_SENTIMENT(
    transcripcion,
    ['servicio al cliente', 'producto', 'precio', 'asesor', 'tiempo de respuesta']
  ) AS sentimiento_por_aspecto
FROM TRANSCRIPCIONES;

In [ ]:
-- Recomendaciones para el asesor con AI_COMPLETE (LLM coaching)
SELECT
  file_name,
  AI_COMPLETE(
    'claude-4-sonnet',
    CONCAT(
      'Eres un coach experto en servicio al cliente. Analiza esta transcripción y genera recomendaciones para el asesor en formato JSON con: ',
      '{"puntos_dolor": [...], "fortalezas_asesor": [...], "areas_mejora": [...], "recomendacion_inmediata": "...", "script_sugerido": "frase exacta a usar en la próxima interacción"}. ',
      'Responde SOLO el JSON, en español. Transcripción: ', transcripcion
    )
  ) AS recomendaciones_asesor
FROM TRANSCRIPCIONES;

In [ ]:
-- Scorecard 360: sentimiento (AI_SENTIMENT) + categoría (AI_CLASSIFY) + nota del asesor (AI_COMPLETE)
SELECT
  file_name AS llamada,
  AI_SENTIMENT(transcripcion):categories[0]:sentiment::VARCHAR AS sentimiento,
  AI_CLASSIFY(
    transcripcion,
    ['Reclamo', 'Consulta', 'Oferta comercial', 'Soporte técnico', 'Otro']
  ):labels[0]::VARCHAR AS categoria,
  AI_COMPLETE(
    'claude-4-sonnet',
    CONCAT(
      'Califica del 1 al 10 al asesor en empatía, resolución y profesionalismo. ',
      'Responde SOLO un JSON: {"empatia": N, "resolucion": N, "profesionalismo": N, "nota_general": N, "veredicto": "texto corto"}. ',
      'Transcripción: ', transcripcion
    )
  ) AS evaluacion_asesor
FROM TRANSCRIPCIONES;

## Ejercicio 4: Cortex Code - Construyamos con lenguaje natural

Ahora viene lo mágico. Vamos a salir del notebook y a usar **Cortex Code** dentro de Snowsight para construir cosas **escribiendo solo en español**. No necesitamos saber SQL avanzado. No necesitamos saber YAML. Solo describir lo que queremos.

### Cómo lo vamos a hacer
1. En Snowsight, abramos **Cortex Code** (icono en el panel lateral o atajo `Cmd/Ctrl + I`)
2. Asegurémonos de estar en el contexto: `HOL_AI_SUMMIT.PUBLIC` con el warehouse `HOL_WH`
3. Copiamos y pegamos cada uno de estos prompts (uno a la vez) y observamos cómo Cortex Code genera el código por nosotros:

**Prompt 1 - Vista que clasifica imágenes:**
> Crea una vista llamada V_IMAGENES_CLASIFICADAS que recorra todos los archivos del stage IMAGENES y agregue una columna con la clasificación del tipo de imagen (cédula, accidente vehicular, factura, otro) usando AI_CLASSIFY sobre AI_COMPLETE multimodal.

**Prompt 2 - Unifiquemos todo en una vista 360:**
> Crea una vista llamada V_HOL_360 que combine las filas de DOCS_PARSED y TRANSCRIPCIONES en un solo dataset con columnas: tipo_fuente, archivo, contenido, sentimiento.

**Prompt 3 - Conversemos con nuestros datos:**
> Genera un SELECT que invoque a AI_COMPLETE preguntándole: ¿cuánto es el canon mensual del contrato número 2 y cuál fue el sentimiento de la última llamada?

**Prompt 4 - Construyamos una app:**
> Crea una app Streamlit in Snowflake llamada DASHBOARD_HOL que muestre el total de imágenes por tipo, el sentimiento de las llamadas en gráfico de barras, y un campo de chat conectado al agente.

Esto demuestra algo enorme: **una persona de negocio puede construir pipelines de IA productivos sin pasar por un sprint de desarrollo**. Eso cambia las reglas del juego.

## Ejercicio 5: Cortex Analyst + Cortex Search + Snowflake Intelligence

Este es el ejercicio estrella del Workshop. Vamos a construir, **desde cero**, los tres componentes que hacen que Snowflake Intelligence sea único en el mercado. Y lo vamos a hacer sin escribir SQL ni YAML manualmente — solo con Cortex Code.

### Lo que vamos a construir
```
┌─────────────────────────────────────────────────────────────┐
│           SNOWFLAKE INTELLIGENCE (nuestro agente)           │
├──────────────────────┬──────────────────────────────────────┤
│  Cortex Analyst      │  Cortex Search                       │
│  (datos estructurados)│  (documentos no estructurados)      │
│                      │                                      │
│  • Pólizas vendidas  │  • Contratos de arrendamiento        │
│  • Clientes          │  • Transcripciones de llamadas       │
│  • Reclamaciones     │                                      │
└──────────────────────┴──────────────────────────────────────┘
```

### Paso 1: Echemos un vistazo a los datos que tenemos

In [ ]:
-- Explorar las tablas de negocio (creadas por setup.sql)
SELECT 'POLIZAS' AS tabla, COUNT(*) AS filas FROM POLIZAS
UNION ALL SELECT 'CLIENTES', COUNT(*) FROM CLIENTES
UNION ALL SELECT 'RECLAMACIONES', COUNT(*) FROM RECLAMACIONES
UNION ALL SELECT 'BASE_CONOCIMIENTO', COUNT(*) FROM BASE_CONOCIMIENTO;

In [ ]:
-- Vista rápida de pólizas vendidas
SELECT fecha, tipo_poliza, producto, region, cliente, vendedor, prima_mensual, estado
FROM POLIZAS ORDER BY fecha DESC LIMIT 10;

### Paso 2: Creemos Cortex Analyst con Cortex Code

Abramos **Cortex Code** (`Cmd/Ctrl + I`) y peguemos este prompt. ¡Tal cual!

> **Prompt - Creemos la Semantic View:**
> Crea una semantic view llamada SV_SEGUROS_DEMO sobre las tablas POLIZAS, CLIENTES y RECLAMACIONES en HOL_AI_SUMMIT.PUBLIC. Incluye dimensiones para tipo_poliza, región, vendedor, segmento de cliente y tipo de siniestro. Define métricas para total de primas, número de pólizas, monto total de reclamaciones aprobadas y tasa de aprobación. Agrega time_dimensions sobre las fechas y relaciones entre las tablas usando el campo cliente/nombre. Incluye verified_queries para: top vendedores por primas, reclamaciones pendientes, y distribución de pólizas por región.

**Un momento, ¿qué es Cortex Analyst?** Es la pieza de Snowflake que toma una pregunta en lenguaje natural y la convierte en SQL correcto. Para que funcione bien, necesita una **Semantic View** que le explique qué significan los datos. Eso es lo que acabamos de crear, **escribiendo en español**.

Después de pegar el prompt, verifiquemos que se creó:

In [ ]:
-- La Semantic View SV_SEGUROS ya fue creada por setup.sql
-- Verificar que existe:
SHOW SEMANTIC VIEWS IN HOL_AI_SUMMIT.PUBLIC;

### Paso 3: Creemos Cortex Search con Cortex Code

Vamos otra vez a **Cortex Code** y pegamos:

> **Prompt - Creemos el Search Service:**
> Crea un Cortex Search Service llamado SEARCH_UNIFICADO sobre la tabla BASE_CONOCIMIENTO en HOL_AI_SUMMIT.PUBLIC. La columna de búsqueda principal es 'contenido', los atributos son 'tipo_documento' y 'file_name'. Usa el warehouse HOL_WH, target_lag de 1 hora, y el embedding model snowflake-arctic-embed-l-v2.0.

**¿Qué hace Cortex Search?** Es búsqueda semántica nativa — lo mismo que haríamos con Pinecone, Weaviate o Elasticsearch + embeddings, pero **sin tener que mover los datos a otro sistema, sin pagar otra licencia y sin sincronizaciones**. Aquí estamos indexando tanto contratos como transcripciones de llamadas en un solo servicio.

Probémoslo:

In [ ]:
-- Probar Cortex Search (el servicio DOCS_SEARCH ya existe del setup)
SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'HOL_AI_SUMMIT.PUBLIC.DOCS_SEARCH',
  '{
    "query": "obligaciones del arrendatario",
    "columns": ["contenido", "tipo_documento", "file_name"],
    "limit": 2
  }'
) AS resultado;

### Paso 4: Construyamos el Agente con Snowflake Intelligence usando Cortex Code

Llegamos al momento estrella. Vamos a **Cortex Code** y pegamos:

> **Prompt - Creemos el Agente:**
> Crea un agente llamado AGENTE_SEGUROS_DEMO en HOL_AI_SUMMIT.PUBLIC que combine:
> 1. Una herramienta cortex_analyst_text_to_sql conectada a la semantic view SV_SEGUROS para consultar datos de pólizas, clientes y reclamaciones
> 2. Una herramienta cortex_search conectada al servicio DOCS_SEARCH para buscar en contratos y transcripciones de llamadas
> 3. Una herramienta data_to_chart para generar gráficos
>
> El agente debe responder en español, es un asistente de una empresa de seguros en Colombia. Las instrucciones de orquestación deben indicar cuándo usar cada herramienta. Incluye 5 sample_questions sobre ventas, contratos, siniestros y sentimiento de llamadas.

**¿Qué es Snowflake Intelligence?** Es la capa conversacional que orquesta todas las herramientas que acabamos de crear. El agente decide solo cuándo usar Analyst (para datos estructurados), cuándo usar Search (para documentos), y cuándo usar gráficos. Nosotros solo preguntamos en lenguaje natural.

### Paso 5: Conversemos con nuestro agente en Snowflake Intelligence

1. Vamos a **AI & ML > Snowflake Intelligence** en Snowsight
2. Seleccionamos **Agente Seguros 360** (creado por el setup)
3. Probemos estas preguntas y observemos cómo el agente decide qué herramienta usar:

| Pregunta | Lo que va a usar | Tipo de información |
|----------|--------------------|-----------------------|
| ¿Cuál es el total de primas por región? | Cortex Analyst | Datos estructurados |
| ¿Qué dice el contrato sobre las cláusulas de terminación? | Cortex Search | Documentos |
| Muéstrame un gráfico de reclamaciones por tipo de siniestro | Analyst + Chart | Datos + Visual |
| ¿Qué ofrecieron en la última llamada telefónica? | Cortex Search | Audio transcrito |
| ¿Quién es el mejor vendedor y cuántas reclamaciones tiene? | Analyst (multi-tabla) | Cruce de datos |

Lo poderoso aquí es que **el agente decide solo qué herramienta usar** según la pregunta. Eso es lo que diferencia a Snowflake Intelligence: una sola interfaz conversacional que combina datos estructurados, documentos, audio e imágenes.

## Cierre - Lo que acabamos de hacer en 20 minutos

| Capacidad | Función / Herramienta | Por qué importa |
|-----------|----------------------|-----------------|
| Análisis de imágenes | AI_COMPLETE multimodal | Sin GPU, sin APIs externas |
| Parsing de documentos | AI_PARSE_DOCUMENT | Soporta PDF, DOCX, PPTX nativo |
| Extracción estructurada | AI_EXTRACT | JSON listo para alimentar nuestro BI |
| Transcripción + sentimiento | AI_TRANSCRIBE + AI_SENTIMENT | Audio multilingüe |
| Búsqueda semántica | CORTEX SEARCH SERVICE | Sin vector DB externa |
| Text-to-SQL inteligente | CORTEX ANALYST + Semantic View | Preguntas en lenguaje natural → SQL |
| Agentes conversacionales | SNOWFLAKE INTELLIGENCE | Combina datos + documentos + audio |
| Visualización automática | data_to_chart | Gráficos sin código |
| Construcción sin código | Cortex Code | Todo construido con prompts en español |

### Lo que nos llevamos de aquí

**Snowflake es la única plataforma donde una persona, en 20 minutos, puede:**
1. Procesar imágenes, documentos y audio sin mover los datos
2. Crear modelos semánticos que entienden nuestro negocio
3. Indexar documentos con búsqueda semántica nativa
4. Construir agentes inteligentes que combinan TODO
5. Hacer lo anterior **escribiendo en español** con Cortex Code

Una sola plataforma. Gobernada. Sin mover datos. Conversacional. Lista para que nuestro equipo la use mañana mismo.

**¡Gracias por participar!** 🚀